IMPOET IMP LIB'S

In [52]:
import os
from dotenv import load_dotenv

# langchain

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain.agents import create_agent


In [13]:
load_dotenv()

True

In [15]:
groq_key=os.getenv("GROQ_API_KEY")
jina_key=os.getenv("JINA_API_KEY")

print("Environment variables loaded successfully.")

Environment variables loaded successfully.


Loading data

In [21]:
DATA_FILE_PATH = r"data\hr_policy.txt"
print(f"Data file path: {DATA_FILE_PATH}")

Data file path: data\hr_policy.txt


Data ingetion

In [ ]:
loader = TextLoader(DATA_FILE_PATH,encoding ="utf-8")
documents = loader.load()
print(f"Number of documents loaded: {len(documents)}")
print("="*40)
print(documents)

Number of documents loaded: 1
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their da

In [27]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


In [28]:
print("total number of characters in the document:", len(documents[0].page_content))

total number of characters in the document: 2597


Data Splitter


In [32]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks=text_splitter.split_documents(documents)
print(f"Number of chunks created: {len(chunks)}")

Number of chunks created: 9


In [37]:
print(chunks[8].page_content)

8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.


Embeddings


In [43]:
embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")
print(" embeddings model loaded successfully.", embeddings_model.model_name)

 embeddings model loaded successfully. jina-embeddings-v2-base-en


# Store data in vector data base


In [46]:
vector_store = FAISS.from_documents( chunks, embeddings_model)
print("Vector store created successfully.")
print("Total number of vectors in the store:", vector_store.index.ntotal)

Vector store created successfully.
Total number of vectors in the store: 9


WE NEVER STORED


In [47]:
test_query = "how many leaves are allowed for a female employee"
# similarity search
top_matches = vector_store.similarity_search(test_query, k=3)
print(f"query : {test_query}")
for i, match in enumerate(top_matches):
    print(f"Match {i+1}:")
    print(f"Content: {match.page_content}")
    print(f"Metadata: {match.metadata}")
    print("="*40)

query : how many leaves are allowed for a female employee
Match 1:
Content: 1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.
Metadata: {'source': 'data\\hr_policy.txt'}
Match 2:
Content: 7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.
Metadata: {'source': 'data\\hr_policy.txt'}
Match 3:
Content: 3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employ

# Data Retrival

In [ ]:
llm=ChatGroq(
    groq_api_key=groq_key,
    model_name="openai/gpt-oss-120b",
    temperature=0.3
)
llm.model_name

'openai/gpt-oss-120b'

In [50]:
test_response = llm.invoke("Is learing rag is hard ? ans in short")

In [51]:
test_response.content

"Learning RAG can be challenging at first, especially if you're new to both retrieval systems and large‑language models. However, with clear tutorials, modular libraries (like LangChain, Haystack, or LlamaIndex), and step‑by‑step practice, most people pick it up fairly quickly. So: **moderately hard at the start, but very doable with the right resources.**"

# Ai Agent

llm-brain
tool-super power
memory-knowledge


In [59]:
# 1. Create retriever
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)


# 2. Create HR policy search tool
def search_hr_policy(question: str) -> str:
    """Search the HR policy documents and return relevant information."""

    matching_chunks = retriever.invoke(question)

    return "\n\n".join(
        chunk.page_content
        for chunk in matching_chunks
    )


# 3. Create the agent
hr_assistant = create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt="""You are a friendly HR assistant.

Always use the search_hr_policy tool to look up the facts before answering.

If the answer isn't in the search results, say "I don't know".
"""
)


# 4. Function to ask the agent
def ask_hr_assistant(question: str) -> str:
    """Send a question to the HR agent and print a nicely formatted answer."""

    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question
                }
            ]
        }
    )

    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()

    return answer


# 5. Ask a question
ask_hr_assistant("How many days of annual leave do employees get?")

QUESTION: How many days of annual leave do employees get?
------------------------------------------------------------
ANSWER: Employees are entitled to **20 days of paid annual leave per calendar year**.



'Employees are entitled to **20 days of paid annual leave per calendar year**.'